# ⛽ Smart Fuel Demand Prediction

**B.Tech CSE (AI/ML) — beginner-friendly project notebook**

This notebook loads the synthetic starter dataset, explores it, creates features, trains multiple regression models, compares them, and saves the best models.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

df = pd.read_csv("../data/fuel_sales.csv", parse_dates=["date"])
df.head()


## 1. Understand the data

The target is the number of litres sold. Calendar, weather and previous-sales features are used as inputs.

In [ ]:
print(df.shape)
print(df.info())
print(df.describe())


## 2. Simple visualization

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(df["date"], df["petrol_sales_litres"], label="Petrol")
plt.plot(df["date"], df["diesel_sales_litres"], label="Diesel")
plt.title("Daily Fuel Sales")
plt.xlabel("Date")
plt.ylabel("Litres")
plt.legend()
plt.show()


## 3. Feature selection

We use a time-based split instead of random shuffling because future dates should not leak into the training data.

In [ ]:
features = ['day_of_week', 'month', 'is_weekend', 'is_holiday', 'temperature_c', 'rainfall_mm', 'petrol_lag_1', 'petrol_lag_7', 'petrol_rolling_7']
target = "petrol_sales_litres"
split = int(len(df)*0.8)

X_train = df[features].iloc[:split]
X_test = df[features].iloc[split:]
y_train = df[target].iloc[:split]
y_test = df[target].iloc[split:]

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))


## 4. Train and compare models

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=250, random_state=42, max_depth=12),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42, n_estimators=200, max_depth=3, learning_rate=0.05)
}

results = []
trained = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append([name, mean_absolute_error(y_test, pred), r2_score(y_test, pred)])
    trained[name] = model

results_df = pd.DataFrame(results, columns=["Model","MAE","R2"]).sort_values("MAE")
results_df


## 5. Select the best model

Lower MAE means the prediction is closer to the real sales value. Higher R² is generally better.

In [ ]:
best_name = results_df.iloc[0]["Model"]
best_model = trained[best_name]
print("Best model:", best_name)

pred = best_model.predict(X_test)

plt.figure(figsize=(12,5))
plt.plot(df["date"].iloc[split:], y_test.values, label="Actual")
plt.plot(df["date"].iloc[split:], pred, label="Predicted")
plt.title("Actual vs Predicted Petrol Demand")
plt.xlabel("Date")
plt.ylabel("Litres")
plt.legend()
plt.show()


## 6. Save the model

In [ ]:
joblib.dump(best_model, "../models/petrol_demand_model.pkl")
print("Petrol model saved.")


## 7. Diesel model

In [ ]:
diesel_features = ['day_of_week', 'month', 'is_weekend', 'is_holiday', 'temperature_c', 'rainfall_mm', 'diesel_lag_1', 'diesel_lag_7', 'diesel_rolling_7']
X_train_d = df[diesel_features].iloc[:split]
X_test_d = df[diesel_features].iloc[split:]
y_train_d = df["diesel_sales_litres"].iloc[:split]
y_test_d = df["diesel_sales_litres"].iloc[split:]

diesel_model = GradientBoostingRegressor(
    random_state=42, n_estimators=200, max_depth=3, learning_rate=0.05
)
diesel_model.fit(X_train_d, y_train_d)
diesel_pred = diesel_model.predict(X_test_d)

print("Diesel MAE:", round(mean_absolute_error(y_test_d, diesel_pred), 2))
print("Diesel R2:", round(r2_score(y_test_d, diesel_pred), 4))

joblib.dump(diesel_model, "../models/diesel_demand_model.pkl")


## 8. Important project note

The included dataset is synthetic and is intended for development/demo use. For a real project, replace it with your own anonymized daily station records and document the data source and privacy handling.